# Xarray-spatial
### User Guide: Hydrology tools
-----
The Hydrology tools provide a complete workflow for analyzing drainage patterns and water flow across a terrain surface. Starting from a digital elevation model (DEM), you can compute flow directions, accumulate flow, delineate watersheds, and extract stream networks.

This guide walks through each tool in the order you would typically use them:

[Sink Detection](#Sink-Detection): Identifies and labels depression cells in a flow direction grid.

[Depression Filling](#Depression-Filling): Removes sinks from a DEM so water can flow continuously to the edges.

[Flow Direction](#Flow-Direction): Computes D8 flow direction for each cell based on steepest descent.

[Multiple Flow Direction (MFD)](#Multiple-Flow-Direction-(MFD)): Partitions flow to all downslope neighbors with an adaptive exponent.

[Flow Accumulation](#Flow-Accumulation): Counts how many upstream cells drain through each cell.

[Drainage Basins](#Drainage-Basins): Labels every cell with the ID of the outlet it drains to.

[Stream Order](#Stream-Order): Assigns Strahler or Shreve stream order to cells in the drainage network.

[Stream Link](#Stream-Link): Segments the stream network into individually labeled links.

[Snap Pour Point](#Snap-Pour-Point): Moves user-placed pour points onto the nearest high-accumulation cell.

[Watershed Delineation](#Watershed-Delineation): Labels every cell with the pour point it drains to.

[Flow Path Tracing](#Flow-Path-Tracing): Traces downstream paths from selected start points to their outlets.

-----------

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

import xrspatial

## Load Elevation Data

We load a section of SRTM-class elevation data from the [Copernicus 30m DEM](https://registry.opendata.aws/copernicus-dem/), a freely accessible global elevation dataset hosted on AWS as Cloud-Optimized GeoTIFFs. The tile below covers the southern Washington Cascades.

If the remote file is unavailable (no network, firewall, etc.), we fall back to xarray-spatial's built-in terrain generator.

In [2]:
try:
    import rasterio
    from rasterio.windows import Window

    url = (
        "https://copernicus-dem-30m.s3.amazonaws.com/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM.tif"
    )

    with rasterio.open(url) as src:
        # Read a 600x600 window from the southern Cascades
        window = Window(col_off=2400, row_off=2400, width=600, height=600)
        data = src.read(1, window=window).astype(np.float64)
        nodata = src.nodata

    if nodata is not None:
        data[data == nodata] = np.nan

    H, W = data.shape
    dem = xr.DataArray(data, dims=['y', 'x'], name='elevation',
                       attrs={'res': (1, 1)})
    dem['y'] = np.linspace(H - 1, 0, H)
    dem['x'] = np.linspace(0, W - 1, W)
    print(f"Loaded Copernicus 30m DEM: {dem.shape}, "
          f"elevation range {np.nanmin(dem.values):.0f} to {np.nanmax(dem.values):.0f} m")

except Exception as e:
    print(f"Remote DEM unavailable ({e}), generating synthetic terrain")
    H, W = 600, 600
    dem = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'])
    dem = dem.xrs.generate_terrain(seed=10)
    dem.name = 'elevation'
    print(f"Generated terrain: {dem.shape}")

Loaded Copernicus 30m DEM: (600, 600), elevation range 564 to 2532 m


### Visualize the DEM

We render a hillshade and drape the elevation colors on top. This base map reappears throughout the guide as context for the hydrology outputs.

In [ ]:
hillshade = dem.xrs.hillshade()

def plot_basemap(ax):
    """Plot hillshade + elevation basemap."""
    ax.imshow(hillshade.values, cmap='gray')
    ax.imshow(dem.values, cmap='terrain', alpha=0.5)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Sink Detection

Real DEMs contain small depressions (sinks) where water would pool instead of flowing to the grid boundary. The `sink` function finds cells with D8 direction code 0 (no downhill neighbor) and groups adjacent ones into labeled depressions using 8-connected component labeling.

We compute a temporary flow direction grid on the raw DEM to show where these sinks are. The red highlights below mark the depressions. In the next section we fill these so that water can drain continuously to the edges.

In [ ]:
# Compute flow direction on the raw DEM to find sinks
flow_dir_raw = xrspatial.flow_direction(dem)
sinks = xrspatial.sink(flow_dir_raw)

n_sink_cells = int(np.sum(~np.isnan(sinks.values)))
n_sink_groups = len(np.unique(sinks.values[~np.isnan(sinks.values)]))
print(f"{n_sink_cells} sink cells in {n_sink_groups} depressions "
      f"({100 * n_sink_cells / (H * W):.1f}% of grid)")

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
sinks_data = np.ma.masked_invalid(sinks.values)
vmin, vmax = np.nanpercentile(sinks.values[np.isfinite(sinks.values)], [2, 98])
ax.imshow(sinks_data, cmap='Reds', alpha=200/255, vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Depression Filling

The `fill` function raises each depression cell to the elevation of its spill point using the [Planchon-Darboux algorithm](https://doi.org/10.1016/S0341-8162(01)00164-3), so that every cell has a path to the grid boundary. This is a standard preprocessing step for DEM-based hydrology.

Parameters:
- **z_limit** (optional): caps the maximum fill depth per cell. Cells that would need a deeper fill revert to their original elevation. This is useful for preserving real lakes or large depressions that you don't want removed.

One subtlety: filling creates flat areas where many adjacent cells share exactly the same elevation. The D8 algorithm assigns code 0 (pit) to these cells because there is no unique steepest neighbor. To resolve flats, we add sub-millimeter random noise after filling. This gives every cell a unique elevation while changing the surface by less than 1 mm.

In [ ]:
dem_filled = xrspatial.fill(dem)

fill_depth = dem_filled - dem
n_filled = int(np.sum(fill_depth.values > 0))
max_depth = np.nanmax(fill_depth.values)
print(f"Filled {n_filled} cells, max fill depth: {max_depth:.2f} m")

# Filling creates flat areas (many cells at exactly the same elevation).
# D8 assigns code 0 to these because there is no unique steepest neighbor.
# Resolve flats by adding sub-mm noise, then re-filling the tiny new pits,
# then adding even finer noise to break any remaining ties.
rng = np.random.RandomState(42)
dem_filled.values += rng.uniform(0, 0.001, dem_filled.shape)
dem_filled = xrspatial.fill(dem_filled)  # re-fill micro-sinks from noise
rng2 = np.random.RandomState(123)
dem_filled.values += rng2.uniform(0, 1e-6, dem_filled.shape)  # break last ties

fill_mask = fill_depth.copy()
fill_mask.values = np.where(fill_depth.values > 0, 1.0, np.nan)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(fill_mask.values)
ax.imshow(masked, cmap='Reds', alpha=200/255)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Flow Direction

The `flow_direction` function assigns each cell a D8 direction code indicating which of its 8 neighbors receives the steepest downhill flow. The encoding uses powers of two:

```
 32  64  128
 16   0    1
  8   4    2
```

Code 0 means the cell is a pit (no downhill neighbor). NaN cells produce NaN output.

Parameters:
- **boundary**: controls edge handling. `'nan'` (default) assigns NaN to edge cells; `'nearest'`, `'reflect'`, and `'wrap'` extend the grid before computing directions.

Here we compute flow direction on the filled (and flat-resolved) DEM. This is the direction grid used by every tool from here on.

In [ ]:
flow_dir = xrspatial.flow_direction(dem_filled)

# Verify that filling + perturbation eliminated interior sinks
sinks_after = xrspatial.sink(flow_dir)
n_sinks_after = int(np.sum(~np.isnan(sinks_after.values)))
print(f"Sinks remaining after fill + perturbation: {n_sinks_after}")

fig, ax = plt.subplots(figsize=(10, 7.5))
fdir_data = flow_dir.values.copy()
vmin, vmax = np.nanpercentile(fdir_data[np.isfinite(fdir_data)], [2, 98])
ax.imshow(fdir_data, cmap='hsv', vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Multiple Flow Direction (MFD)

D8 sends all flow to a single neighbor, which works well for channelized flow but not for dispersive hillslope flow. The `flow_direction_mfd` function partitions flow from each cell to **all** downslope neighbors. An adaptive exponent from Qin et al. (2007) adjusts per-cell: steep convergent terrain concentrates flow (closer to D8), while gentle slopes spread it out.

The output is a 3-D DataArray `(8, H, W)` where each band holds the fraction of flow directed to one of the 8 neighbors (E, SE, S, SW, W, NW, N, NE). Fractions sum to 1.0 at each cell.

Parameters:
- **p**: flow-partition exponent. `None` (default) uses the adaptive exponent. A positive float sets a fixed exponent (e.g. `p=1.0` for Quinn et al. 1991).
- **boundary**: same edge-handling options as `flow_direction`.

In [ ]:
mfd_fracs = xrspatial.flow_direction_mfd(dem_filled)
print(f"MFD output shape: {mfd_fracs.shape}")
print(f"Dims: {mfd_fracs.dims}")
print(f"Neighbors: {list(mfd_fracs.coords['neighbor'].values)}")

# Show the maximum fraction per cell as a measure of flow concentration.
# Values near 1.0 = almost all flow to one neighbor (D8-like).
# Values near 0.125 = flow spread nearly evenly across 8 neighbors.
import matplotlib.pyplot as plt

max_frac = mfd_fracs.max(dim='neighbor')
n_receivers = (mfd_fracs > 0).sum(dim='neighbor').astype(float)
n_receivers = n_receivers.where(~np.isnan(mfd_fracs.isel(neighbor=0)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_frac.plot(ax=axes[0], cmap='YlOrRd', vmin=0, vmax=1)
axes[0].set_title('Max flow fraction (higher = more concentrated)')

n_receivers.plot(ax=axes[1], cmap='viridis', vmin=0, vmax=8)
axes[1].set_title('Number of receiving neighbors')

plt.tight_layout()
plt.show()

### Comparing D8, D-infinity, and MFD

The three flow direction algorithms represent different trade-offs between simplicity and physical realism:

- **D8**: all flow goes to the single steepest neighbor. Produces clean, deterministic channels but can't represent dispersive hillslope flow.
- **D-infinity** (Tarboton 1997): flow direction is a continuous angle toward the steepest downslope facet. Splits flow between at most two neighbors. Better for smooth surfaces but still limited to two receivers.
- **MFD** (Qin et al. 2007): partitions flow to all downslope neighbors. Most realistic for hillslope processes but produces 8 output bands instead of one.

Below we compare all three on the same DEM.

In [ ]:
# D-infinity flow direction (continuous angle in radians)
flow_dir_dinf = xrspatial.flow_direction_dinf(dem_filled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# D8: color-code the 9 direction codes
shade_d8 = flow_dir.copy()
shade_d8.plot(ax=axes[0], cmap='hsv', add_colorbar=True)
axes[0].set_title('D8 direction code\n(one of 9 discrete values)')

# Dinf: continuous angle 0-2pi
flow_dir_dinf.plot(ax=axes[1], cmap='hsv', vmin=0, vmax=2*np.pi, add_colorbar=True)
axes[1].set_title('D-infinity angle (radians)\n(continuous 0 to 2pi)')

# MFD: show max fraction as a proxy for how concentrated the flow is
max_frac.plot(ax=axes[2], cmap='YlOrRd', vmin=0, vmax=1, add_colorbar=True)
axes[2].set_title('MFD max fraction\n(1.0 = all flow to one neighbor)')

for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Flow direction comparison: D8 vs D-infinity vs MFD')
plt.tight_layout()
plt.show()

# Key differences in a small summary
d8_codes = flow_dir.values[~np.isnan(flow_dir.values)]
mfd_max = max_frac.values[~np.isnan(max_frac.values)]
mfd_max_nonzero = mfd_max[mfd_max > 0]

print(f"D8: {len(np.unique(d8_codes))} unique direction codes (always 1 receiver)")
print(f"Dinf: continuous angle, splits flow between at most 2 neighbors")
print(f"MFD: median max fraction = {np.median(mfd_max_nonzero):.3f}, "
      f"mean receivers = {(n_receivers.values[~np.isnan(n_receivers.values)]).mean():.1f}")

In [ ]:
# Compare MFD exponent modes
mfd_p1 = xrspatial.flow_direction_mfd(dem_filled, p=1.0)
mfd_p8 = xrspatial.flow_direction_mfd(dem_filled, p=8.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, data, title in zip(
    axes,
    [mfd_p1.max(dim='neighbor'), max_frac, mfd_p8.max(dim='neighbor')],
    ['p=1.0 (dispersive)', 'Adaptive (Qin 2007)', 'p=8.0 (concentrated)'],
):
    data.plot(ax=ax, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('MFD exponent comparison: lower p spreads flow, higher p concentrates it')
plt.tight_layout()
plt.show()

## Flow Accumulation

For each cell, `flow_accumulation` counts how many upstream cells drain through it (including itself). The result spans several orders of magnitude: hilltops have accumulation 1, while river channels collect thousands of upstream cells.

This is the workhorse of the hydrology stack. Most downstream tools (stream order, stream link, snap pour point) use the accumulation grid as input. Thresholding it is the standard way to extract a stream network.

In [ ]:
flow_accum = xrspatial.flow_accumulation(flow_dir)

print(f"Accumulation range: {np.nanmin(flow_accum.values):.0f} to "
      f"{np.nanmax(flow_accum.values):.0f}")

# Log-scale rendering reveals the full drainage network
water_cmap = LinearSegmentedColormap.from_list('water', ['white', '#08306b'])
data = flow_accum.values.copy()
positives = data[data > 0]
vmin = max(np.nanmin(positives), 1e-10)

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.imshow(data, cmap=water_cmap, norm=LogNorm(vmin=vmin))
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Extract a stream network by thresholding accumulation.
# Lower thresholds give denser networks; higher thresholds keep only major channels.
threshold = 200
streams = flow_accum.copy()
streams.values = np.where(streams.values >= threshold, streams.values, np.nan)

n_stream_cells = int(np.sum(~np.isnan(streams.values)))
print(f"Stream cells (accum >= {threshold}): {n_stream_cells}")

stream_cmap = LinearSegmentedColormap.from_list('water', ['lightblue', 'darkblue'])
stream_data = streams.values.copy()
stream_positives = stream_data[stream_data > 0]
stream_vmin = max(np.nanmin(stream_positives), 1e-10)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(stream_data)
ax.imshow(masked, cmap=stream_cmap, alpha=220/255, norm=LogNorm(vmin=stream_vmin))
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Flow Accumulation (MFD)

`flow_accumulation_mfd` takes the 3-D fractional output from
`flow_direction_mfd` and routes upstream contributing area through all
downslope paths at once.  Where D8 accumulation produces sharp, single-pixel
drainage lines, MFD accumulation spreads flow across the landscape and
produces smoother contributing-area fields.

In [ ]:
flow_accum_mfd = xrspatial.flow_accumulation_mfd(mfd_fracs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# D8 accumulation (log scale)
im0 = axes[0].imshow(np.log1p(flow_accum.values), cmap='Blues')
axes[0].set_title('D8 flow accumulation (log)')
fig.colorbar(im0, ax=axes[0], shrink=0.6)

# MFD accumulation (log scale)
im1 = axes[1].imshow(np.log1p(flow_accum_mfd.values), cmap='Blues')
axes[1].set_title('MFD flow accumulation (log)')
fig.colorbar(im1, ax=axes[1], shrink=0.6)

plt.tight_layout()
plt.show()

## Drainage Basins

The `basin` function automatically identifies every outlet in the D8 grid (pits and cells that flow off the grid edge), assigns each a unique ID, and labels every cell with the ID of the outlet it drains to.

No pour points needed: the algorithm finds all outlets on its own. This is useful for getting a quick overview of the drainage structure before deciding where to place pour points for more targeted watershed analysis.

In [ ]:
basins = xrspatial.basin(flow_dir)

n_basins = len(np.unique(basins.values[~np.isnan(basins.values)]))
print(f"Found {n_basins} drainage basins")

basin_cmap = LinearSegmentedColormap.from_list('basin', ['blue', 'green', 'yellow', 'orange', 'red', 'purple'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(hillshade.values, cmap='gray')
basin_data = basins.values.copy()
vmin, vmax = np.nanpercentile(basin_data[np.isfinite(basin_data)], [2, 98])
masked = np.ma.masked_invalid(basin_data)
ax.imshow(masked, cmap=basin_cmap, alpha=150/255, vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Stream Order

The `stream_order` function classifies stream cells by their position in the drainage hierarchy. Two methods are available:

- **Strahler** (default): a headwater stream is order 1. When two streams of the same order meet, the downstream segment increments by one. When streams of different order meet, the higher order continues unchanged.
- **Shreve**: stream magnitude equals the sum of all upstream headwaters. Every confluence adds the magnitudes of its tributaries.

Parameters:
- **threshold**: minimum flow accumulation for a cell to be classified as a stream. Lower values produce a denser network; higher values keep only major channels.
- **method**: `'strahler'` or `'shreve'`.

In [ ]:
strahler = xrspatial.stream_order(
    flow_dir, flow_accum, threshold=200, method='strahler'
)
print(f"Max Strahler order: {int(np.nanmax(strahler.values))}")

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(strahler.values)
ax.imshow(masked, cmap=stream_cmap, alpha=220/255)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
shreve = xrspatial.stream_order(
    flow_dir, flow_accum, threshold=200, method='shreve'
)
print(f"Max Shreve magnitude: {int(np.nanmax(shreve.values))}")

shreve_data = shreve.values.copy()
vmin, vmax = np.nanpercentile(shreve_data[np.isfinite(shreve_data)], [2, 98])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(shreve_data)
ax.imshow(masked, cmap=stream_cmap, alpha=220/255, vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Stream Link

The `stream_link` function assigns a unique integer ID to each contiguous stream segment between junctions, headwaters, and outlets. This is useful for per-reach statistics (length, slope, contributing area) or for linking stream topology to a graph data structure.

Parameters:
- **threshold**: same meaning as in `stream_order` -- minimum accumulation for a cell to be part of the stream network.

In [ ]:
links = xrspatial.stream_link(flow_dir, flow_accum, threshold=200)

link_ids = np.unique(links.values[~np.isnan(links.values)])
print(f"Found {len(link_ids)} stream link segments")

link_cmap = LinearSegmentedColormap.from_list('link', ['cyan', 'green', 'yellow', 'orange', 'red'])
link_data = links.values.copy()
vmin, vmax = np.nanpercentile(link_data[np.isfinite(link_data)], [2, 98])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(link_data)
ax.imshow(masked, cmap=link_cmap, alpha=200/255, vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Snap Pour Point

Users typically place pour points by hand, but those clicks rarely land exactly on the drainage channel. The `snap_pour_point` function moves each pour point to the highest flow-accumulation cell within a circular search radius, so that downstream `watershed` calls delineate correctly.

Parameters:
- **search_radius**: maximum distance in pixels (Euclidean) to search for a higher-accumulation cell. Larger radii are more forgiving of placement error but risk jumping to the wrong channel.

Below, we place three pour points slightly off the drainage network (red dots) and snap them onto the channels (green dots).

In [ ]:
H, W = dem.shape

pour_points = xr.DataArray(
    np.full((H, W), np.nan, dtype=np.float64),
    dims=dem.dims, coords=dem.coords,
)

# Find the peak-accumulation cell in three quadrants, then offset by
# a few pixels to simulate hand-placed pour points that miss the channel.
accum_vals = flow_accum.values.copy()
accum_vals[np.isnan(accum_vals)] = 0

quadrants = [
    (slice(20, H // 2 - 20), slice(20, W // 2 - 20)),
    (slice(20, H // 2 - 20), slice(W // 2 + 20, W - 20)),
    (slice(H // 2 + 20, H - 20), slice(20, W // 2 - 20)),
]

for label, (ys, xs) in enumerate(quadrants, start=1):
    sub = accum_vals[ys, xs]
    lr, lc = np.unravel_index(sub.argmax(), sub.shape)
    r = min(ys.start + lr + 5, H - 1)
    c = max(xs.start + lc - 3, 0)
    pour_points.values[r, c] = float(label)

snapped = xrspatial.snap_pour_point(
    flow_accum, pour_points, search_radius=10
)

# Get row, col positions for scatter plotting
pp_rows, pp_cols = np.where(~np.isnan(pour_points.values))
sn_rows, sn_cols = np.where(~np.isnan(snapped.values))

stream_data = streams.values.copy()
stream_positives = stream_data[stream_data > 0]
stream_vmin = max(np.nanmin(stream_positives), 1e-10)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked_streams = np.ma.masked_invalid(stream_data)
ax.imshow(masked_streams, cmap='Blues', alpha=100/255, norm=LogNorm(vmin=stream_vmin))
ax.scatter(pp_cols, pp_rows, c='red', s=50, zorder=5, label='Original')
ax.scatter(sn_cols, sn_rows, c='lime', s=50, zorder=5, label='Snapped')
ax.legend(loc='upper right')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Watershed Delineation

The `watershed` function traces every cell downstream through the D8 flow direction grid until it reaches a pour point (or runs out of grid). Each cell is labeled with the pour point it drains to.

This is the targeted version of `basin`: instead of automatically finding all outlets, you supply specific pour points and get back the contributing area for each one. Using `snap_pour_point` first ensures the pour points sit on the drainage network.

In [ ]:
ws = xrspatial.watershed(flow_dir, snapped)

ws_ids = np.unique(ws.values[~np.isnan(ws.values)])
print(f"Delineated {len(ws_ids)} watersheds")

ws_cmap = LinearSegmentedColormap.from_list('ws', ['blue', 'green', 'yellow'])
ws_data = ws.values.copy()
vmin, vmax = np.nanpercentile(ws_data[np.isfinite(ws_data)], [2, 98])

stream_positives = stream_data[stream_data > 0]
stream_vmin = max(np.nanmin(stream_positives), 1e-10)

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(hillshade.values, cmap='gray')
masked_ws = np.ma.masked_invalid(ws_data)
ax.imshow(masked_ws, cmap=ws_cmap, alpha=150/255, vmin=vmin, vmax=vmax)
masked_streams = np.ma.masked_invalid(stream_data)
ax.imshow(masked_streams, cmap='Blues', alpha=120/255, norm=LogNorm(vmin=stream_vmin))
ax.scatter(sn_cols, sn_rows, c='red', s=50, zorder=5, label='Pour points')
ax.legend(loc='upper right')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Flow Path Tracing

The `flow_path` function does the opposite of `watershed`: given a set of start points, it follows the D8 direction grid downstream from each start, marking every cell along the way with that start point's label. Paths terminate at pits, NaN cells, or the grid edge.

This is the single most useful visualization for exploring where water goes from a particular location. Below, we pick five tributary cells (moderate accumulation) spread across the grid and trace their downstream paths to the major channels and grid boundary.

In [ ]:
start_points = xr.DataArray(
    np.full((H, W), np.nan, dtype=np.float64),
    dims=dem.dims, coords=dem.coords,
)

# Pick tributary cells (moderate accumulation) spread across the grid.
# These are already in small channels, so their paths run all the way
# to the major rivers and grid edge.
accum_vals = flow_accum.values.copy()
accum_vals[np.isnan(accum_vals)] = 0
trib_mask = (accum_vals >= 50) & (accum_vals <= 200)
trib_rows, trib_cols = np.where(trib_mask)

# Spread the 5 start points across the grid by dividing into quadrants + center
regions = [
    (slice(20, H // 3), slice(20, W // 3)),             # top-left
    (slice(20, H // 3), slice(2 * W // 3, W - 20)),     # top-right
    (slice(H // 3, 2 * H // 3), slice(W // 3, 2 * W // 3)),  # center
    (slice(2 * H // 3, H - 20), slice(20, W // 3)),     # bottom-left
    (slice(2 * H // 3, H - 20), slice(2 * W // 3, W - 20)),  # bottom-right
]
label = 1
for ys, xs in regions:
    in_region = (
        (trib_rows >= ys.start) & (trib_rows < ys.stop) &
        (trib_cols >= xs.start) & (trib_cols < xs.stop)
    )
    if np.any(in_region):
        idx = np.where(in_region)[0][len(np.where(in_region)[0]) // 2]
        start_points.values[trib_rows[idx], trib_cols[idx]] = float(label)
        label += 1

paths = xrspatial.flow_path(flow_dir, start_points)

n_path_cells = int(np.sum(~np.isnan(paths.values)))
n_starts = int(np.sum(~np.isnan(start_points.values)))
print(f"Traced {n_path_cells} path cells from {n_starts} start points")

path_cmap = LinearSegmentedColormap.from_list('path', ['red', 'orange', 'yellow', 'lime', 'cyan'])
path_data = paths.values.copy()
vmin, vmax = np.nanpercentile(path_data[np.isfinite(path_data)], [2, 98])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
masked = np.ma.masked_invalid(path_data)
ax.imshow(masked, cmap=path_cmap, alpha=220/255, vmin=vmin, vmax=vmax)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## References

- Jenson, S.K. and Domingue, J.O. (1988). Extracting Topographic Structure from Digital Elevation Data for Geographic Information System Analysis. *Photogrammetric Engineering and Remote Sensing*, 54(11), 1593-1600.
- Planchon, O. and Darboux, F. (2001). A fast, simple and versatile algorithm to fill the depressions of digital elevation models. *Catena*, 46(2-3), 159-176.
- Quinn, P., Beven, K., Chevallier, P., and Planchon, O. (1991). The prediction of hillslope flow paths for distributed hydrological modelling using digital terrain models. *Hydrological Processes*, 5(1), 59-79.
- Qin, C., Zhu, A.X., Pei, T., Li, B., Zhou, C., and Yang, L. (2007). An adaptive approach to selecting a flow-partition exponent for a multiple-flow-direction algorithm. *International Journal of Geographical Information Science*, 21(4), 443-458.
- Strahler, A.N. (1957). Quantitative analysis of watershed geomorphology. *Transactions of the American Geophysical Union*, 38(6), 913-920.
- Tarboton, D.G. (1997). A new method for the determination of flow directions and upslope areas in grid digital elevation models. *Water Resources Research*, 33(2), 309-319.
- Copernicus DEM on AWS: https://registry.opendata.aws/copernicus-dem/
